## Functions

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import sys, os
from ipywidgets import interact
from IPython.display import display, Math, Latex
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display, Math
from scipy.special import genlaguerre, legendre, lpmv
from matplotlib import cm
from scipy.special import sph_harm, genlaguerre, factorial
import matplotlib as mpl
%matplotlib inline

In [5]:
def xsph(theta, phi, r):
    return np.cos(phi)*np.sin(theta)*r

def ysph(theta, phi, r):
    return np.sin(phi)*np.sin(theta)*r

def zsph(theta, phi, r):
    return np.cos(theta)*r


def az(phi,m):
    return np.exp(1j*m*phi)
def polar(theta, l, m):
    return lpmv(abs(m), l, np.cos(theta))

def ang_norm(l,m):
    num = (2*l+1)*factorial(l-abs(m))
    den = 4*np.pi*factorial(l+abs(m))
    return (num/den)**.5

def Ylm(theta, phi, l, m):
    sign = 1
    if not (m==0):
        sign = (-1)**m
    return ang_norm(l,m)*az(phi,m)*polar(theta,l,m)*sign

a0 = 5.29e-11 #m
def norm(n,l,Z=1):
    return (factorial(n-l-1)/(2*n*factorial(n+l)))**.5 * (2*Z/(n*a0))**(l+1.5)

def Rnl(r, n, l, Z=1):
    a0 = 5.29e-11 #m
    rho = 2*Z*r/(n*a0)
    gl = genlaguerre(n-l-1,2*l+1)(rho)
    p1 = r**l * np.exp(-Z*r/(n*a0))
    return norm(n,l,Z)*gl*p1 
cmap = LinearSegmentedColormap.from_list(
    "bright_div",
    [[1,0,0], "black", [0,.3,1]]
)

def psi_combo(theta,phi,l,m, combo):
    mp = Ylm(theta,phi,l,abs(m))
    mm = Ylm(theta,phi,l,-abs(m))
    if combo=='plus':
        return np.real((mp + mm)/(2**.5))
    elif combo=='minus':
        return np.real((mp - mm)/(1j*2**.5))
    elif combo=='None':
        return np.real(mp)

In [14]:
combo_dict = {
    's':  {'l': 0, 'm': 0, 'combo': 'plus', 'name':'s'},

    'px': {'l': 1, 'm': 1, 'combo': 'plus', 'name':'p_x'},
    'py': {'l': 1, 'm': 1, 'combo': 'minus', 'name':'p_y'},
    'pz': {'l': 1, 'm': 0, 'combo': 'None', 'name':'p_z'},

    'dz^2':       {'l': 2, 'm': 0, 'combo': 'None', 'name':'d_{{z^2}}'},
    'dxz':        {'l': 2, 'm': 1, 'combo': 'plus', 'name':'d_{{xz}}'},
    'dyz':        {'l': 2, 'm': 1, 'combo': 'minus', 'name':'d_{{yz}}'},
    'dxy':        {'l': 2, 'm': 2, 'combo': 'minus', 'name':'d_{{xy}}'},
    'dx^2-y^2':   {'l': 2, 'm': 2, 'combo': 'plus', 'name':'d_{{x^2-y^2}}'},
}

def slater_zeff(Z, electron="valence"):
    """
    Estimate Zeff using Slater's rules for a neutral atom.

    Parameters
    ----------
    Z : int
        Atomic number.
    electron : "valence" or tuple
        "valence" estimates Zeff for the outermost electron.
        Or specify an electron as (n, subshell), e.g. (2, "p"), (3, "d").

    Returns
    -------
    dict with configuration, target electron, shielding S, and Zeff.
    """

    subshell_order = [
        (1, "s", 2),(2, "s", 2), (2, "p", 6),(3, "s", 2), (3, "p", 6),(4, "s", 2),
        (3, "d", 10),(4, "p", 6),(5, "s", 2),(4, "d", 10),(5, "p", 6),(6, "s", 2),
        (4, "f", 14),(5, "d", 10),(6, "p", 6),(7, "s", 2),(5, "f", 14),(6, "d", 10),(7, "p", 6),
    ]

    if Z < 1 or Z > 118:
        raise ValueError("Z must be between 1 and 118.")

    # Build aufbau electron configuration
    remaining = Z
    config = []
    for n, l, cap in subshell_order:
        if remaining <= 0:
            break
        occ = min(cap, remaining)
        config.append((n, l, occ))
        remaining -= occ

    # Choose target electron
    if electron == "valence":
        n_target, l_target, _ = config[-1]
    else:
        n_target, l_target = electron

    # Check that target exists
    target_occ = None
    for n, l, occ in config:
        if n == n_target and l == l_target:
            target_occ = occ
            break

    if target_occ is None:
        
        raise ValueError(f"No electrons in {n_target}{l_target} for Z={Z}.")

    S = 0.0

    for n, l, occ in config:
        if n == n_target and l == l_target:
            occ -= 1  # do not shield the electron from itself

        if occ <= 0:
            continue

        # Target is s or p electron
        if l_target in ["s", "p"]:
            if n == n_target:
                if n_target == 1:
                    S += 0.30 * occ
                else:
                    S += 0.35 * occ
            elif n == n_target - 1:
                if l in ["s", "p"]:
                    S += 0.85 * occ
                else:
                    S += 1.00 * occ
            elif n <= n_target - 2:
                S += 1.00 * occ

        # Target is d or f electron
        elif l_target in ["d", "f"]:
            if n == n_target and l == l_target:
                S += 0.35 * occ
            elif n < n_target:
                S += 1.00 * occ

    Zeff = Z - S

    return {
        "Z": Z,
        "configuration": config,
        "target": f"{n_target}{l_target}",
        "shielding": S,
        "Zeff": Zeff,
    }

In [ ]:
def diatom():
    nwidget = widgets.IntSlider(min=1, max=4, value=2)
    lwidget = widgets.IntSlider(min=0, max=3, value=1)
    mwidget = widgets.IntSlider(min=-3, max=3, value=0)
    orbwidget = widgets.Dropdown(options = ['px', 'py', 'pz'])

    savewidget = widgets.ToggleButton()
    @interact(
        part = widgets.RadioButtons(options=['Complex','Real', 'Imaginary'], value='Real', description='Real or Imaginary'),
        prob = widgets.RadioButtons(options=['Wavefunction', 'Probability Density'], value='Wavefunction',description='Wavefunction or Prob.'),
        Z = widgets.IntSlider(min=1, max=14, value=6),
        OrbType = widgets.RadioButtons(options=['Linear Combination', 'Ylm']),
        l_type = orbwidget,
        n = nwidget,
        l = lwidget,
        m = mwidget,
        d = widgets.FloatSlider(min=0, max=5, value=1.098, step=.01, description='Distance'),
        elevation = widgets.FloatSlider(min=0, max=90, value=90, step=1),
        rotate = widgets.FloatSlider(min=0, max=360, value=90, step=1),
        invert = widgets.ToggleButton(value=False),
        add = widgets.RadioButtons(options=['add','subtract'],value='add', description='Combination')
    )
    def g(**kargs):
        s_keys = ['s']
        p_keys = ['px', 'py', 'pz']
        d_keys = ['dz^2','dxz','dyz', 'dxy','dx^2-y^2']
        if kargs['OrbType']=='Linear Combination':
            mwidget.disabled = True
            orbwidget.disabled=False
        else:
            mwidget.disabled = False
            orbwidget.disabled=True

        kargs['transparency']=.3
        l = kargs['l']
        m =kargs['m']

        if l > kargs['n']-1:
            nwidget.value = l + 1
            return 'l > n-1'
        if m > kargs['l']:
            mwidget.value = kargs['l']
            return 'm > l'
        if m < -kargs['l']:
            mwidget.value = -kargs['l']
            return 'm < -l'
        mwidget.max=kargs['l']
        mwidget.min=-kargs['l']
        n = kargs['n']

        

        if l == 0:
            orbwidget.options = s_keys
        if l== 1:
            orbwidget.options = p_keys
        if l==2:
            orbwidget.options = d_keys
        
        scale = int(kargs['n']**2 * 4)/kargs['Z']
        extent = 5*a0#scale*a0*2
        x = np.linspace(-extent, extent, 50)
        X,Y,Z = np.meshgrid(x,x,x)
        d = kargs['d']*1e-10

        R1 = ((X)**2+Y**2+(Z+d/2)**2)**.5
        Theta1 = np.arccos((Z+d/2)/R1)
        Phi1 = np.atan2(Y,X)

        R2 = ((X)**2+Y**2+(Z-d/2)**2)**.5
        Theta2 = np.arccos((Z-d/2)/R2)
        Phi2 = np.atan2(Y,X)

        z_val_d = slater_zeff(kargs['Z'])
        nval = int(z_val_d['target'][0])
        if n > nval:
            z_eff_d = slater_zeff(kargs['Z'], (nval, "s"))
        else:
            z_eff_d = slater_zeff(kargs['Z'], (n, "s"))
        zeff = z_eff_d['Zeff']
        display(Latex(f'$Z_{{eff}} = {zeff:.1f}$'))
        if kargs['OrbType']=='Ylm':
            psi1 = Ylm(Theta1, Phi1, l,m)*Rnl(R1, n, l, zeff)
            psi2 = Ylm(Theta2, Phi2, l,m)*Rnl(R2, n, l, zeff)
            wfa = f'\\psi_{{A,{n},{l},{m}}}'
            wfb = f'\\psi_{{B,{n},{l},{m}}}'
        else:
            m=combo_dict[kargs['l_type']]['m']
            psi1 = psi_combo(Theta1, Phi1, l, m, combo_dict[kargs['l_type']]['combo'])*Rnl(R1,n,l,zeff)
            psi2 = psi_combo(Theta2, Phi2, l, m, combo_dict[kargs['l_type']]['combo'])*Rnl(R2,n,l,zeff)
            lname = combo_dict[kargs['l_type']]['name']
            wfa = f'{n}{lname}{{}}_A'
            wfb = f'{n}{lname}{{}}_B'
        

        if kargs['add']=='add':
            display(Latex(f'$\\psi = \\dfrac{{1}}{{\\sqrt{{2(1+S)}}}}({wfa}+{wfb})$'))
            
            
            psi = psi1 + psi2
            
        else:
            psi = psi1 - psi2
            display(Latex(f'$\\psi = \\dfrac{{1}}{{\\sqrt{{2(1-S)}}}}({wfa}-{wfb})$'))

        prob1 = np.real(psi1 * np.conj(psi1))
        prob2 = np.real(psi2 * np.conj(psi2))
        prob = np.real(psi * np.conj(psi))
        integral = prob.sum()/(prob1.sum())
        if kargs['add']=='add':
            S = (integral-2)/2
        else:
            S = -(integral-2)/2
        display(Latex(f'$S = {S:.2f}$'))

        psi*=(1/integral)**.5
        mx = np.max(np.real(psi))
        inds = np.argwhere(abs(psi) > .05 * mx)
        


        X_plot = X[inds[:,0], inds[:,1], inds[:,2]].ravel()
        Y_plot = Y[inds[:,0], inds[:,1], inds[:,2]].ravel()
        Z_plot = Z[inds[:,0], inds[:,1], inds[:,2]].ravel()
        psi_plot = psi[inds[:,0], inds[:,1], inds[:,2]].ravel()

        if kargs['prob']=='Probability Density':
            psi_plot = np.real(psi_plot * np.conj(psi_plot))
            colors = np.zeros((len(psi_plot), 4))  
            alpha = psi_plot/np.max(psi_plot)
            colors[:, 3] = alpha
        elif kargs['prob']=='Wavefunction':
            if kargs['part']=='Real':
                psi_plot = np.real(psi_plot)
                reds = np.zeros_like(psi_plot)
                reds[np.argwhere(psi_plot>0)] = .8
                blues = np.zeros_like(psi_plot)
                blues[np.argwhere(psi_plot<=0)] = .8
                alpha = abs(psi_plot)/np.max(abs(psi_plot))*kargs['transparency']
                colors = np.zeros((len(psi_plot), 4))  
                colors[:, 0] = reds
                colors[:, 2] = blues
                colors[:, 3] = alpha
            elif kargs['part']=='Imaginary':
                psi_plot = np.imag(psi_plot)
                reds = np.zeros_like(psi_plot)
                reds[np.argwhere(psi_plot>0)] = .8
                blues = np.zeros_like(psi_plot)
                blues[np.argwhere(psi_plot<=0)] = .8
                alpha = abs(psi_plot)/np.max(abs(psi_plot))*kargs['transparency']
                colors = np.zeros((len(psi_plot), 4))  
                colors[:, 0] = reds
                colors[:, 2] = blues
                colors[:, 3] = alpha
            elif kargs['part']== 'Complex':
                angles = np.angle(psi_plot)+np.pi/4
                norm = plt.Normalize(-np.pi, np.pi)
                colors = cm.hsv(norm(angles))
                alpha = abs(psi_plot)/np.max(abs(psi_plot))*kargs['transparency']
                colors[:, 3] = alpha
        fig= plt.figure()
        ax = fig.add_subplot(projection='3d')
        if kargs['invert']:
            ax.scatter( -Z_plot, -X_plot, -Y_plot, c=colors, rasterized=True)
        else:
            ax.scatter( Z_plot, X_plot, Y_plot, c=colors, rasterized=True)
        ax.scatter( d/2,0,0, c='k')
        ax.scatter( -d/2,0,0, c='k')
        ax.view_init(elev=kargs['elevation'], azim=kargs['rotate'])
        ax.set_xlim(-extent, extent)
        ax.set_ylim(-extent, extent)
        ax.set_zlim(-extent, extent)
        ax.set_aspect('equal')
        ax.set_axis_off()
        plt.show()


## LCAO Molecular Orbitals

In [21]:
diatom()

interactive(children=(RadioButtons(description='Real or Imaginary', index=1, options=('Complex', 'Real', 'Imag…